# BirdCLEF+ 2026 — Perch v2 Submission (Overlap Inference)

Perch v2 TFLite + MLP/LogReg による提出用ノートブック。
**2.5秒オーバーラップ推論**で窓境界の取りこぼしを改善。

## Changes from previous submission
- 推論窓: 5秒ごと（重複なし） → **2.5秒ずつスライド（50%オーバーラップ）**
- 同一5秒バケットに属する複数窓の予測を**平均**して最終予測とする
- MLP / LogReg どちらのモデルにも対応

## Kaggle Settings
| Setting | Value |
|---------|-------|
| Accelerator | None (CPU) |
| Internet | **OFF** |

In [ ]:
import os, glob, pickle, collections, time, warnings
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# ============================================================
# Path + Config
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): COMP_DATA = p; break
assert COMP_DATA

TFLITE_PATH = None
for f in glob.glob('/kaggle/input/**/*.tflite', recursive=True): TFLITE_PATH = f; break
assert TFLITE_PATH, 'perch_v2.tflite not found'

# Support both MLP (.pth) and LogReg (.pkl)
MLP_PATH = None
for f in glob.glob('/kaggle/input/**/*mlp*.pth', recursive=True): MLP_PATH = f; break

CLF_PATH = None
for f in glob.glob('/kaggle/input/**/*classifier*.pkl', recursive=True): CLF_PATH = f; break
for f in glob.glob('/kaggle/input/**/*mlp*le*.pkl', recursive=True): CLF_PATH = f; break

# Find label encoder
LE_PATH = None
for f in glob.glob('/kaggle/input/**/*.pkl', recursive=True):
    if 'le' in os.path.basename(f).lower() or 'label' in os.path.basename(f).lower() or 'encoder' in os.path.basename(f).lower():
        LE_PATH = f; break
if LE_PATH is None: LE_PATH = CLF_PATH

SAMPLE_RATE = 32000
DURATION = 5
WINDOW_SAMPLES = SAMPLE_RATE * DURATION
OVERLAP_SECONDS = 2.5  # 50% overlap
STEP_SAMPLES = int(SAMPLE_RATE * OVERLAP_SECONDS)
OUTPUT_DIR = '/kaggle/working'

print(f'Data: {COMP_DATA}')
print(f'TFLite: {TFLITE_PATH}')
print(f'MLP: {MLP_PATH}')
print(f'CLF/LE: {CLF_PATH}')
print(f'Overlap: {OVERLAP_SECONDS}s (step={STEP_SAMPLES} samples)')

In [ ]:
# ============================================================
# Load TFLite Perch
# ============================================================
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH, num_threads=4)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Find embedding output
interpreter.set_tensor(input_details[0]['index'], np.zeros((1, WINDOW_SAMPLES), dtype=np.float32))
interpreter.invoke()
EMBED_IDX = None
for od in output_details:
    r = interpreter.get_tensor(od['index'])
    if r.shape[-1] == 1536: EMBED_IDX = od['index']; break
if EMBED_IDX is None:
    sizes = [(od['index'], interpreter.get_tensor(od['index']).shape[-1]) for od in output_details]
    EMBED_IDX = max(sizes, key=lambda x: x[1])[0]

def embed_fn(waveform):
    interpreter.set_tensor(input_details[0]['index'], waveform.reshape(1,-1).astype(np.float32))
    interpreter.invoke()
    emb = interpreter.get_tensor(EMBED_IDX)
    if emb.ndim > 2: emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
    elif emb.ndim == 2: emb = emb[0]
    return emb.astype(np.float32)

print(f'Embedding shape: {embed_fn(np.zeros(WINDOW_SAMPLES, dtype=np.float32)).shape}')

In [ ]:
# ============================================================
# Load classifier (MLP or LogReg)
# ============================================================
import torch
import torch.nn as nn

USE_MLP = MLP_PATH is not None

if USE_MLP:
    class MLPClassifier(nn.Module):
        def __init__(self, input_dim, hidden_dim, num_classes, dropout):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
                nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))
        def forward(self, x): return self.net(x)
        @torch.no_grad()
        def predict_proba(self, x):
            self.eval()
            return torch.softmax(self.forward(x), dim=1).cpu().numpy()
    
    ckpt = torch.load(MLP_PATH, map_location='cpu')
    mlp = MLPClassifier(1536, ckpt['hidden_dim'], ckpt['n_classes'], ckpt['dropout'])
    mlp.load_state_dict(ckpt['model_state_dict'])
    mlp.eval()
    n_classes = ckpt['n_classes']
    print(f'MLP loaded: {n_classes} classes')
    
    with open(LE_PATH, 'rb') as f: le = pickle.load(f)
    
    def predict_fn(emb):
        t = torch.tensor(emb, dtype=torch.float32).unsqueeze(0)
        return mlp.predict_proba(t)[0]
else:
    with open(CLF_PATH, 'rb') as f: data = pickle.load(f)
    clf, le = data['clf'], data['le']
    n_classes = len(le.classes_)
    print(f'LogReg loaded: {n_classes} classes')
    
    def predict_fn(emb):
        raw = clf.predict_proba(emb.reshape(1,-1))[0]
        full = np.zeros(n_classes, dtype=np.float32)
        for i, c in enumerate(clf.classes_): full[c] = raw[i]
        return full

# Species mapping
sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']
species_to_idx = {}
for sp in species_cols:
    try: species_to_idx[sp] = int(le.transform([sp])[0])
    except ValueError: pass
print(f'Mapped: {len(species_to_idx)} / {len(species_cols)} species')

In [ ]:
# ============================================================
# Overlap Inference on test soundscapes
# ============================================================
test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.endswith('.ogg'))
print(f'Test files: {len(test_files)}')
print(f'Window: {DURATION}s, Step: {OVERLAP_SECONDS}s (overlap={DURATION-OVERLAP_SECONDS}s)')

# bucket_key -> list of prediction arrays
bucket_preds = collections.defaultdict(list)
t0 = time.time()

for fi, fname in enumerate(test_files):
    path = os.path.join(test_dir, fname)
    stem = os.path.splitext(fname)[0]
    full_wf, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(full_wf) == 0: continue
    
    # Slide with overlap
    start = 0
    while start < len(full_wf):
        window = full_wf[start:start + WINDOW_SAMPLES]
        if len(window) < WINDOW_SAMPLES:
            window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
        
        # Map to 5-second bucket
        bucket_idx = start // WINDOW_SAMPLES
        bucket_sec = bucket_idx * DURATION
        row_id = f'{stem}_{bucket_sec}'
        
        emb = embed_fn(window.astype(np.float32))
        probs = predict_fn(emb)
        bucket_preds[row_id].append(probs)
        
        start += STEP_SAMPLES
    
    if (fi + 1) % 50 == 0:
        print(f'  [{fi+1}/{len(test_files)}] {int(time.time()-t0)}s', flush=True)

print(f'\nDone: {int(time.time()-t0)}s, buckets={len(bucket_preds)}')
# Show how many windows per bucket (should be ~2 with 50% overlap)
if bucket_preds:
    avg_windows = np.mean([len(v) for v in bucket_preds.values()])
    print(f'Average windows per bucket: {avg_windows:.1f}')

In [ ]:
# ============================================================
# Build submission (average overlapping predictions)
# ============================================================
rows = []
for row_id, pred_list in bucket_preds.items():
    avg = np.mean(pred_list, axis=0)  # average overlapping windows
    row = {'row_id': row_id}
    for sp in species_cols:
        row[sp] = float(avg[species_to_idx[sp]]) if sp in species_to_idx else 0.0
    rows.append(row)

submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)
submission = submission.set_index('row_id').reindex(sample_sub['row_id'], fill_value=0.0).reset_index()
submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)

assert len(submission) == len(sample_sub)
assert list(submission.columns) == list(sample_sub.columns)
assert submission.isnull().sum().sum() == 0

vals = submission.drop('row_id', axis=1).values
print(f'Submission saved')
print(f'  Rows: {len(submission)}')
print(f'  Mean: {vals.mean():.6f}')
print(f'  Max: {vals.max():.4f}')
print(f'  Zero: {(vals == 0).mean():.2%}')